In [9]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, precision_recall_curve

In [10]:
def optimal_threshold(ai_model, X_train, y_train, X_early_test, y_early_test, min_prec, n_folds=3):
    """Takes multiple practice exams and finds the best safety threshold"""
    
    # Pack up the practice quiz so cross_val_predict can use it
    practice_quiz = {'eval_set': [(X_early_test, y_early_test)]}
    
    # Step A: Take 3 practice exams (now properly handing it the practice quiz)
    # Get fraud probability
    cv_pred_prob = cross_val_predict(
        ai_model, X_train, y_train, 
        method='predict_proba', 
        cv=n_folds, 
        params=practice_quiz
    )[:,1]
    
    # Step B: Test every possible dial setting
    precisions, _, threshold = precision_recall_curve(y_train, cv_pred_prob)
    precisions = precisions[:-1] 

    return min(threshold[precisions >= min_prec])

In [11]:
def thresholded_predict(X_test, ai_model, threshold):
    """Makes a final prediction using our custom threshold instead of 50%"""
    # Get the probability (confidence percentage) for the final exam questions
    return np.array([1 if (prediction >= threshold) else 0 for prediction in ai_model.predict_proba(X_test)[:,1]])

In [12]:
if __name__ == "__main__":
    # 1. Load the Data
    print("Loading data...")
    import kagglehub
    import os

    path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
    csv_file_path = os.path.join(path, 'creditcard.csv')
    data = pd.read_csv(csv_file_path)
    X = data.drop('Class', axis=1) # Remove the Class - Answer (0 or 1 / Legit or Fraud)
    y = data['Class'] # The answer sheet

    # 3. Split the data
    # Final Test

    # Stratify: Split the fraud and normal equal percentage in Training and Testing Set
    # random_state: Way of shuffle to prevent everytime got diff shuffle techniques
    # 100% take out 20% (test) 80% (training)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

    # Early Test

    # 80% take out 10% (test) 70% (training)
    # Conclusion: 72% - Trainining, 20% - Final Test, 8% - Early Test
    X_train, X_early_test, y_train, y_early_test = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=1)

    # ==========================================
    # 4. 🔥 TWEAKING SETTINGS 🔥
    # ==========================================
    my_settings = {
        # Up to how many trees the model can build
        # The more the better, BUT waste of time
        # Implement `early_stopping_rounds` so will exit early if useless tree is build
        'n_estimators': 2000,

        # How many "Yes/No" questions each tree can ask in a row.
        # The more the better, BUT waste of time + Overfitting
        'max_depth': 4,

        # How much the AI "corrects" its mistakes after every lesson.
        'learning_rate': 0.05,

        # Tells the AI that 1 Fraud case is worth "X" Normal cases.
        'scale_pos_weight': 400,

        'subsample': 0.8,          # Forces the AI to only look at 80% of rows at a time (reduces noise)
        'colsample_bytree': 0.8,   # Forces the AI to only look at 80% of the clues (prevents obsession)
        'min_child_weight': 4,    # Require at least 4 sources before believing a story
        'reg_alpha': 0.1,         # Cut out a tiny bit of clutter
        'reg_lambda': 1.0,        # Keep the volume balanced

        # Loop for model to re-take early_test.
        # Every one tree build, will have 1 round early_test.
        # If hit personal best, will reset -> 0.
        # If after 50 round, no improve on personal best, then remove the last 50 trees
        # The higher the better, BUT WASTE OF TIME
        'early_stopping_rounds': 50
    }
    
    # MIN_PRECISION is to tune the precision of the model of flagging a transaction as Fraud
    # The lower the MIN_PRECISION, the higher the RECALL_RATE
    MIN_PRECISION = 0.05 # Try 0.05 and 0.99
    # ==========================================

    # 5. Build and Train the Model
    print("Training the AI (this might take a minute)...")
    ai_model = XGBClassifier(**my_settings)

    ai_model.fit(
        X_train, y_train, 
        eval_set=[(X_early_test, y_early_test)], 
        verbose=False # True if want see the log
    )
    print(f"✅ AI stopped studying after {ai_model.best_iteration} trees.")

    # ==========================================
    # FINDING THE PERFECT THRESHOLD
    # ==========================================
    print("\nTaking multiple practice exams (Cross-Validation) to find the perfect dial setting...")
    perfect_threshold = optimal_threshold(ai_model, X_train, y_train, X_early_test, y_early_test, min_prec=MIN_PRECISION, n_folds=3)
    print(f"✅ Perfect Threshold Found! Instead of 50%, we will flag fraud at {perfect_threshold * 100:.2f}% confidence.")

    # ==========================================
    # 6. Take the Final Exam! (Using the new Custom Threshold)
    # ==========================================
    print("\nTesting the AI on the Final Exam...")

    # Instead of the default 50% split, use our perfect custom threshold
    predictions = thresholded_predict(X_test, ai_model, perfect_threshold)

    # 7. Print the Report Card
    print("\n--- 📊 REPORT CARD ---")
    # ----------------------------------------------------
    # 1. ACCURACY: The "Useless" Metric in Fraud Detection
    # What it asks: Out of EVERYTHING you guessed, how many did you get right?
    # Why it's bad: If the AI just guesses "Normal" for everything, it gets 99.8% 
    #               because 99.8% of transactions are normal! It ignores the rare fraud.
    # ----------------------------------------------------
    print(f"Accuracy:  {accuracy_score(y_test, predictions) * 100:.2f}%")
    # ----------------------------------------------------
    # 2. PRECISION: The "False Alarm" Checker - Sensitivity
    # What it asks: When the AI screams "FRAUD!" (Label transaction as FRAUD), what are the chances it is actually right?
    # What it means: A low score means the bank is annoying tons of innocent customers (easy to label transaction as FRAUD)
    #                with false alarms. A high score means no false alarms.
    #                Out of X transaction analyzed, only `PRECISION_PERCENTAGE` is TRUE FRAUD (Confusion Matrix: FALSE POSITIVE)   
    # Our Goal: As long as it stays safely above our minimum limit, we are happy.
    # ----------------------------------------------------
    print(f"Precision: {precision_score(y_test, predictions) * 100:.2f}%")
    # ----------------------------------------------------
    # 3. RECALL: The "Slipping Through" Checker
    # What it asks: Out of ALL the real fraudsters, how many did the AI successfully catch?
    # What it means: A low score means fraudsters are slipping past security. 
    #                A high score means we caught almost all of them.
    # The Relationship: Precision and Recall fight like a seesaw. Because we let our 
    #                   Precision drop so low (lots of false alarms), our Recall should 
    #                   shoot way up to catch the bad guys! (Confusion Matrix: TRUE POSITIVE)
    # ----------------------------------------------------
    print(f"Recall:    {recall_score(y_test, predictions) * 100:.2f}% (How much did we catch?)")

Loading data...
Using Colab cache for faster access to the 'creditcardfraud' dataset.
Training the AI (this might take a minute)...
✅ AI stopped studying after 660 trees.

Taking multiple practice exams (Cross-Validation) to find the perfect dial setting...
[0]	validation_0-logloss:0.49498
[1]	validation_0-logloss:0.46578
[2]	validation_0-logloss:0.43821
[3]	validation_0-logloss:0.41442
[4]	validation_0-logloss:0.39172
[5]	validation_0-logloss:0.37027
[6]	validation_0-logloss:0.35065
[7]	validation_0-logloss:0.33207
[8]	validation_0-logloss:0.31501
[9]	validation_0-logloss:0.29898
[10]	validation_0-logloss:0.28329
[11]	validation_0-logloss:0.26941
[12]	validation_0-logloss:0.25651
[13]	validation_0-logloss:0.24509
[14]	validation_0-logloss:0.23269
[15]	validation_0-logloss:0.22138
[16]	validation_0-logloss:0.21044
[17]	validation_0-logloss:0.19949
[18]	validation_0-logloss:0.18934
[19]	validation_0-logloss:0.18033
[20]	validation_0-logloss:0.17134
[21]	validation_0-logloss:0.16273
[22]